# Groundwater feature integration

This adapts the team's existing merge workflow. The nitrate table remains the base, with one output row per original sample. Soil and terrain features are per well; climate features use the well's nearest PRISM grid point and the sample date.

The old merge workflow also combined estimated well depth and surface-water nitrate. Those are excluded here. No input data is edited. The final cell writes a new validation CSV only if that filename is unused.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / '.git').exists() and (folder / 'notebooks').is_dir():
        PROJECT_ROOT = folder
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the project repository.")

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping
import fiona

GROUNDWATER = PROJECT_ROOT / "notebooks" / "groundwater"
SOILS = PROJECT_ROOT / "notebooks" / "soils"
ELEVATION = PROJECT_ROOT / "notebooks" / "elevation"
CLIMATE = PROJECT_ROOT / "notebooks" / "precipitation"
NITRATE_PATH = GROUNDWATER / "groundwater_nitrate_clean.csv"
BUFFER_PATH = GROUNDWATER / "groundwater_well_buffers_500m.gpkg"
SOIL_PATH = PROJECT_ROOT / 'data' / 'soils' / 'soils_huc8_attributes.gpkg'
DEM_PATH = PROJECT_ROOT / 'data' / 'elevation' / 'dem_watershed_10m.tif'
PRISM_PATH = CLIMATE / "prism_climate_monthly_clean.csv"
CROP_PATH = PROJECT_ROOT / "notebooks" / "crop_mapping" / "well_crop_features_all.csv"
METRIC_CRS = "EPSG:3310"

def checked_left_merge(base, features, key, feature_name):
    """Left-join features while preserving the base rows and their order."""
    if key not in base.columns or key not in features.columns:
        raise KeyError(f"{feature_name}: missing join key {key!r}.")
    if features[key].duplicated().any():
        raise ValueError(f"{feature_name}: feature keys are not unique.")

    original_ids = base["observation_id"].copy()
    original_rows = len(base)
    merged = base.merge(
        features,
        on=key,
        how="left",
        validate="many_to_one",
        sort=False,
        indicator="_feature_match",
    )
    if len(merged) != original_rows or not merged["observation_id"].equals(original_ids):
        raise AssertionError(f"{feature_name}: merge changed base rows or order.")

    matched = int(merged["_feature_match"].eq("both").sum())
    coverage = {
        "feature": feature_name,
        "matched_rows": matched,
        "unmatched_rows": original_rows - matched,
        "missing_fraction": (original_rows - matched) / original_rows,
    }
    return merged.drop(columns="_feature_match"), coverage


## 1. Check the inputs

The checks stop if a required file or column is missing. That keeps an unavailable feature from being silently skipped.

In [ ]:
required_paths = [NITRATE_PATH, BUFFER_PATH, SOIL_PATH, DEM_PATH, PRISM_PATH, CROP_PATH]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing inputs:\n" + "\n".join(map(str, missing_paths)))

observations = pd.read_csv(
    NITRATE_PATH,
    dtype={"gm_well_id": "string", "src_samp_collection_time": "string"},
    parse_dates=["gm_samp_collection_date"],
)
required = {"gm_well_id", "gm_samp_collection_date", "nitrate_mg_L", "gm_latitude", "gm_longitude"}
if not required.issubset(observations.columns):
    raise ValueError(f"Missing nitrate columns: {required - set(observations.columns)}")
if observations.empty:
    raise ValueError("The nitrate table is empty.")
sample_years = observations["gm_samp_collection_date"].dt.year
if not sample_years.between(2022, 2025).all():
    raise ValueError("Expected Charlsey's filtered nitrate data from 2022 through 2025.")
expected_rows = len(observations)
if observations[["gm_well_id", "gm_samp_collection_date"]].isna().any().any():
    raise ValueError("The nitrate table has a missing well ID or sample date.")

observations = observations.copy()
# The canonical CSV row number distinguishes repeated measurements.
observations["source_csv_row"] = observations.index + 2
observations["observation_id"] = "GW-CSV-" + observations["source_csv_row"].astype(str).str.zfill(5)
if observations["observation_id"].duplicated().any():
    raise ValueError("Observation IDs must be unique.")
base_ids = observations["observation_id"].copy()
print("Nitrate rows:", len(observations))
print("Unique wells:", observations.gm_well_id.nunique())
print("Years:", observations.gm_samp_collection_date.dt.year.min(),
      "to", observations.gm_samp_collection_date.dt.year.max())
print("Columns:", observations.columns.tolist())

## 2. Area-weighted SSURGO features

Existing 500 m buffers are overlaid with SSURGO polygons. Numeric values are averaged by intersected area. Category percentages use mapped category area; coverage percentages show how much of the full buffer is mapped. An absent category is 0 when category data are mapped, while no category coverage remains missing. No minimum coverage cutoff is assumed.

In [ ]:
import shapely
buffers = gpd.read_file(BUFFER_PATH)
soils = gpd.read_file(SOIL_PATH)

if buffers.crs is None or soils.crs is None:
    raise ValueError("Both buffer and soil layers need a CRS.")
if buffers.empty:
    raise ValueError("The well buffer layer is empty.")
if buffers.gm_well_id.duplicated().any():
    raise ValueError("Expected one existing buffer per well ID.")

# Check geometries before reprojection or spatial operations. Invalid
# geometries are reported rather than silently repaired.
def check_geometries(frame, layer_name, id_column=None):
    missing = frame.geometry.isna()
    empty = frame.geometry.is_empty
    invalid = ~frame.geometry.is_valid
    problem = missing | empty | invalid

    if problem.any():
        examples = frame.loc[problem, id_column].head(10).tolist() if id_column else frame.index[problem].tolist()[:10]
        raise ValueError(
            f"{layer_name} has {int(problem.sum())} missing, empty, or invalid "
            f"geometries. Examples: {examples}"
        )

    print(f"{layer_name}: all {len(frame)} geometries are valid.")

check_geometries(buffers, "Buffers", "gm_well_id")
check_geometries(soils, "SSURGO soils")

soil_columns = ["slopegradwta", "brockdepmin", "aws0150wta", "drclassdcd", "hydgrpdcd"]
if not set(soil_columns).issubset(soils.columns):
    raise ValueError(f"Missing soil columns: {set(soil_columns) - set(soils.columns)}")

category_fields = ["drclassdcd", "hydgrpdcd"]
category_values = {}
for name in category_fields:
    category_values[name] = sorted(soils[name].dropna().unique(), key=str)

# Area calculations use the same metric CRS for both layers.
buffers = buffers.to_crs("EPSG:3310")
soils = soils.to_crs("EPSG:3310").reset_index(drop=True)

numeric_fields = ["slopegradwta", "brockdepmin", "aws0150wta"]
for name in numeric_fields:
    soils[name] = pd.to_numeric(soils[name], errors="coerce")

soil_features_by_batch = []
batch_size = 100
total_buffers = len(buffers)

for batch_start in range(0, total_buffers, batch_size):
    batch_end = min(batch_start + batch_size, total_buffers)
    batch = buffers.iloc[batch_start:batch_end][["gm_well_id", "geometry"]]

    # Find only soil polygons touching each buffer, then intersect all pairs
    # in one vectorized Shapely operation for this batch.
    candidates = gpd.sjoin(
        batch,
        soils[soil_columns + ["geometry"]],
        how="inner",
        predicate="intersects",
    )

    if candidates.empty:
        pairs = candidates.copy()
        pairs["_intersection_area"] = pd.Series(dtype="float64")
    else:
        soil_positions = candidates["index_right"].to_numpy(dtype=int)
        intersection_geometries = shapely.intersection(
            candidates.geometry.array,
            soils.geometry.array.take(soil_positions),
        )
        intersection_areas = shapely.area(intersection_geometries)
        keep = np.isfinite(intersection_areas) & (intersection_areas > 0)

        pairs = candidates.loc[keep].copy()
        pairs["_intersection_area"] = intersection_areas[keep]

    feature_batch = pd.DataFrame({"gm_well_id": batch["gm_well_id"].to_numpy()})
    buffer_areas = pd.Series(
        batch.geometry.area.to_numpy(),
        index=batch["gm_well_id"].to_numpy(),
    )

    mapped_area = pairs.groupby("gm_well_id")["_intersection_area"].sum()
    mapped_area_for_wells = feature_batch.gm_well_id.map(mapped_area).fillna(0.0)
    if (mapped_area_for_wells.to_numpy() > buffer_areas.to_numpy() * 1.01).any():
        raise ValueError("SSURGO polygons overlap within at least one well buffer.")
    feature_batch["soil_mapped_coverage_pct"] = (
        100 * mapped_area_for_wells.to_numpy() / buffer_areas.to_numpy()
    )

    for name in numeric_fields:
        valid_pairs = pairs.loc[
            pairs[name].notna(), ["gm_well_id", name, "_intersection_area"]
        ]
        valid_area = valid_pairs.groupby("gm_well_id")["_intersection_area"].sum()
        weighted_sum = (
            valid_pairs[name] * valid_pairs["_intersection_area"]
        ).groupby(valid_pairs["gm_well_id"]).sum()

        feature_batch[f"soil_{name}"] = (
            feature_batch.gm_well_id.map(weighted_sum / valid_area).to_numpy()
        )
        feature_batch[f"soil_{name}_coverage_pct"] = (
            100
            * feature_batch.gm_well_id.map(valid_area).fillna(0.0).to_numpy()
            / buffer_areas.to_numpy()
        )

    for name in category_fields:
        valid_pairs = pairs.dropna(subset=[name])
        valid_area = valid_pairs.groupby("gm_well_id")["_intersection_area"].sum()
        valid_area_for_wells = feature_batch.gm_well_id.map(valid_area)

        feature_batch[f"soil_{name}_coverage_pct"] = (
            100
            * valid_area_for_wells.fillna(0.0).to_numpy()
            / buffer_areas.to_numpy()
        )

        category_area = valid_pairs.groupby(
            ["gm_well_id", name], observed=True
        )["_intersection_area"].sum()

        for category in category_values[name]:
            label = str(category).strip().lower().replace(" ", "_").replace("/", "_")
            try:
                area_for_category = category_area.xs(category, level=name)
            except KeyError:
                area_for_category = pd.Series(dtype="float64")

            area_by_well = feature_batch.gm_well_id.map(area_for_category).fillna(0.0)
            percentage = 100 * area_by_well / valid_area_for_wells
            feature_batch[f"soil_{name}_pct_{label}"] = percentage.where(
                valid_area_for_wells > 0
            ).to_numpy()

    soil_features_by_batch.append(feature_batch)
    print(f"Processed SSURGO buffers {batch_start + 1}-{batch_end} of {total_buffers}")

    del candidates, pairs
    if "intersection_geometries" in locals():
        del intersection_geometries
    import gc
    gc.collect()

soil_features = pd.concat(soil_features_by_batch, ignore_index=True)
if soil_features.gm_well_id.duplicated().any():
    raise ValueError("SSURGO feature table is not unique by well.")

print("SSURGO feature rows:", len(soil_features))
print("Average mapped buffer coverage (%):", soil_features.soil_mapped_coverage_pct.mean().round(1))


In [ ]:
# NHD diagnostics paused while river distance is excluded.
# print(type(flowlines))
# print(flowlines.columns)
# print("geometry" in flowlines.columns)

## 3. Terrain and larger-stream distance

The terrain notebook reports DEM CRS EPSG:4269 and pixel spacing near 0.0000926 degrees. The nominal file name is 10 m; exact ground resolution varies by latitude. Distances and buffer areas use EPSG:3310 metres. Topographic position is point elevation minus mean valid elevation in the existing 500 m buffer.

In [ ]:
coordinates = observations[["gm_well_id", "gm_longitude", "gm_latitude"]].drop_duplicates()
if coordinates.gm_well_id.duplicated().any():
    raise ValueError("A well ID has conflicting coordinates.")
well_points = gpd.GeoDataFrame(
    coordinates,
    geometry=gpd.points_from_xy(coordinates.gm_longitude, coordinates.gm_latitude),
    crs="EPSG:4326",
).to_crs(METRIC_CRS)
buffer_geometry = buffers.set_index("gm_well_id").geometry

dem_features = []
with rasterio.open(DEM_PATH) as dem:
    print("DEM CRS:", dem.crs, "pixel size:", dem.res, "NoData:", dem.nodata)
    points_dem = well_points.to_crs(dem.crs)
    point_values = [np.ma.asarray(next(dem.sample([(p.x, p.y)], masked=True))[0]).filled(np.nan) for p in points_dem.geometry]
    for well_id, point_value in zip(well_points.gm_well_id, point_values):
        polygon = gpd.GeoSeries([buffer_geometry.loc[well_id]], crs=METRIC_CRS).to_crs(dem.crs).iloc[0]
        pixels, _ = mask(dem, [mapping(polygon)], crop=True, filled=False)
        valid = pixels[0].compressed()
        valid = valid[np.isfinite(valid)]
        point_value = float(point_value) if np.isfinite(point_value) else np.nan
        mean_nearby = float(valid.mean()) if len(valid) else np.nan
        dem_features.append({"gm_well_id": well_id, "elevation_m": point_value,
                             "topographic_position": point_value - mean_nearby})
dem_features = pd.DataFrame(dem_features)

# distance_to_river_m is temporarily excluded: the available NHD VAA
# streamorder values are all missing, so the order > 3 filter cannot be applied.
# Keep this code for use when populated stream-order data are available.
# gdb_files = [p for p in RAW.rglob("*.gdb") if "NHD" in p.name.upper()]
# if not gdb_files:
#     raise FileNotFoundError(f"No NHD geodatabase found under configured raw folder {RAW}.")
# nhd_path = gdb_files[0]
# layers = fiona.listlayers(nhd_path)
# # NHDFlowline contains the lines; NHDFlowlineVAA is an attribute table.
# flowline_layer = next((layer for layer in layers if layer.lower() == "nhdflowline"), None)
# if flowline_layer is None:
#     raise ValueError(f"No NHDFlowline geometry layer found. Available layers: {layers}")
# flowlines = gpd.read_file(nhd_path, layer=flowline_layer)
# if not isinstance(flowlines, gpd.GeoDataFrame) or flowlines.crs is None:
#     raise ValueError("NHDFlowline must contain geometry and a defined CRS.")
#
# order_names = {"streamorde", "streamorder"}
# order_field = next((col for col in flowlines.columns if col.lower() in order_names), None)
# if order_field is None:
#     # Attach stream order by the exact NHD identifier, preserving line geometry.
#     attribute_layer = next((layer for layer in layers if layer.lower() == "nhdflowlinevaa"), None)
#     if attribute_layer is None:
#         raise ValueError("NHDFlowline has no stream order, and NHDFlowlineVAA is missing.")
#     flowline_attributes = gpd.read_file(nhd_path, layer=attribute_layer)
#     order_field = next((col for col in flowline_attributes.columns if col.lower() in order_names), None)
#     if order_field is None:
#         raise ValueError("NHDFlowlineVAA has no stream-order field.")
#     join_key = "permanent_identifier"
#     for table in [flowlines, flowline_attributes]:
#         if join_key not in table.columns or table[join_key].isna().any():
#             raise ValueError("Both NHD layers need non-missing permanent_identifier values.")
#     flowlines = flowlines.merge(
#         flowline_attributes[[join_key, order_field]],
#         on=join_key,
#         how="left",
#         validate="one_to_one",
#         sort=False,
#     )
#
# orders = pd.to_numeric(flowlines[order_field], errors="coerce")
# if not orders.notna().any():
#     raise ValueError(
#         "NHD geometry loaded successfully, but no matched flowline has a valid "
#         "stream-order value. This input cannot support the stream order > 3 filter; "
#         "provide NHD data with populated stream order. Stream level is not a substitute."
#     )
# large_streams = flowlines.loc[orders > 3].to_crs(METRIC_CRS)
# if large_streams.empty:
#     raise ValueError("NHD has no flowlines with stream order above 3.")
# nearest_river = gpd.sjoin_nearest(
#     well_points[["gm_well_id", "geometry"]],
#     large_streams[["geometry"]],
#     how="left",
#     distance_col="distance_to_river_m",
# )
# nearest_river = nearest_river.sort_values(["gm_well_id", "distance_to_river_m"]).drop_duplicates("gm_well_id")
# dem_features = dem_features.merge(nearest_river[["gm_well_id", "distance_to_river_m"]],
#                                   on="gm_well_id", validate="one_to_one")

## 4. Date-safe PRISM climate features

A well is assigned to its nearest PRISM grid point in EPSG:3310. Features use only complete months before the nitrate sample month. Historical means use complete calendar years before the sample year. Inches are converted to millimetres; °F is converted to °C. Earlier nitrate samples may lack PRISM history because this source begins in 2013.

In [ ]:
climate = pd.read_csv(PRISM_PATH, parse_dates=["date"])
needed = {"point_id", "longitude", "latitude", "date", "precip_in", "tmean_f"}
if not needed.issubset(climate.columns):
    raise ValueError(f"Missing PRISM columns: {needed - set(climate.columns)}")
if climate.duplicated(["point_id", "date"]).any():
    raise ValueError("Duplicate PRISM point-month records found.")
climate["precip_mm"] = climate.precip_in * 25.4
climate["tmean_c"] = (climate.tmean_f - 32) * 5 / 9
climate["year"] = climate.date.dt.year
climate["month"] = climate.date.dt.month
grid = climate[["point_id", "longitude", "latitude"]].drop_duplicates()
grid = gpd.GeoDataFrame(grid, geometry=gpd.points_from_xy(grid.longitude, grid.latitude), crs="EPSG:4326").to_crs(METRIC_CRS)
grid_match = gpd.sjoin_nearest(well_points[["gm_well_id", "geometry"]], grid[["point_id", "geometry"]],
                               how="left", distance_col="prism_grid_distance_m")
grid_match = grid_match.sort_values(["gm_well_id", "prism_grid_distance_m", "point_id"]).drop_duplicates("gm_well_id")
point_for_well = grid_match.set_index("gm_well_id").point_id
by_point = {key: group.sort_values("date").copy() for key, group in climate.groupby("point_id")}

def climate_for_date(point_id, sample_date):
    result = {
        "historical_mean_annual_precip_mm": np.nan,
        "previous_3_complete_months_precip_mm": np.nan,
        "previous_12_complete_months_precip_mm": np.nan,
        "historical_mean_temperature_c": np.nan,
        "previous_3_complete_months_temperature_c": np.nan,
        "previous_12_complete_months_temperature_c": np.nan,
    }
    point_data = by_point.get(point_id)
    if point_data is None:
        return result
    sample_month = sample_date.to_period("M")
    prior = point_data.loc[point_data.date.dt.to_period("M") < sample_month].copy()
    prior["period"] = prior.date.dt.to_period("M")
    for window in [3, 12]:
        periods = pd.period_range(end=sample_month - 1, periods=window, freq="M")
        values = prior.set_index("period").reindex(periods)
        if values.date.notna().all():
            result[f"previous_{window}_complete_months_precip_mm"] = values.precip_mm.sum()
            result[f"previous_{window}_complete_months_temperature_c"] = values.tmean_c.mean()
    prior["year"] = prior.date.dt.year
    full_years = prior.groupby("year").month.nunique()
    full_years = full_years[full_years == 12].index
    history = prior[prior.year.isin(full_years)]
    if not history.empty:
        result["historical_mean_annual_precip_mm"] = history.groupby("year").precip_mm.sum().mean()
        result["historical_mean_temperature_c"] = history.tmean_c.mean()
    return result

climate_rows = []
for row in observations[["gm_well_id", "gm_samp_collection_date"]].itertuples(index=False):
    climate_rows.append(climate_for_date(point_for_well.get(row.gm_well_id),
                                         row.gm_samp_collection_date))
climate_features = pd.DataFrame(climate_rows)
climate_features.insert(0, "observation_id", observations.observation_id.to_numpy())
print("Climate feature missingness:")
display(climate_features.isna().mean().to_frame("missing_fraction"))

## 5. Gavin's 2023 crop features for 2024–2025 comparison

The source table describes 2023. Original column names are retained, but the values are attached only to nitrate observations from 2024–2025. They do not represent earlier historical land use. Unmatched wells remain with missing crop values.

In [ ]:
crop = pd.read_csv(CROP_PATH, dtype={"well_id": "string"})
crop_columns = [
    "pct_veg_berry_500m_2023", "pct_grain_field_pasture_500m_2023",
    "pct_idle_fallow_500m_2023", "pct_cropland_500m_2023",
    "crop_cycles_500m_2023", "pct_vineyard_1000m_2023",
    "pct_grain_field_pasture_1000m_2023", "pct_idle_fallow_1000m_2023",
    "pct_multicrop_1000m_2023", "dist_veg_berry_m_2023",
]
if "well_id" not in crop.columns or not set(crop_columns).issubset(crop.columns):
    raise ValueError(f"Missing crop columns: {set(crop_columns) - set(crop.columns)}")
if crop.well_id.duplicated().any():
    raise ValueError("Crop table must have one row per well_id.")
crop = crop[["well_id", *crop_columns]].rename(columns={"well_id": "gm_well_id"})
recent = observations.loc[observations.gm_samp_collection_date.dt.year.isin([2024, 2025])]
crop_comparison_2024_2025 = recent.merge(crop, on="gm_well_id", how="left", validate="many_to_one")
print("2024–2025 comparison rows:", len(crop_comparison_2024_2025))
print("Wells in crop table:", crop.gm_well_id.nunique())

## 6. Assemble and check the table

Static soil and terrain features join by well ID. Climate and crop features join by observation ID. Every join is left-sided and checked to preserve all original samples. Missing values are reported, not filled.

In [ ]:
integrated = observations.copy()
integrated, soil_match = checked_left_merge(integrated, soil_features, "gm_well_id", "SSURGO")
integrated, dem_match = checked_left_merge(integrated, dem_features, "gm_well_id", "terrain")

crop_rows = integrated[["observation_id", "gm_well_id", "gm_samp_collection_date"]].merge(
    crop, on="gm_well_id", how="left", validate="many_to_one")
is_recent = crop_rows.gm_samp_collection_date.dt.year.isin([2024, 2025])
crop_rows.loc[~is_recent, crop_columns] = np.nan
crop_rows["crop_map_reference_year"] = np.where(is_recent, 2023, np.nan)
integrated, crop_match = checked_left_merge(
    integrated, crop_rows[["observation_id", *crop_columns, "crop_map_reference_year"]],
    "observation_id", "2023 crop comparison")
integrated, climate_match = checked_left_merge(
    integrated, climate_features, "observation_id", "PRISM climate")

if len(integrated) != expected_rows or not integrated.observation_id.equals(base_ids):
    raise AssertionError("The integration changed or reordered nitrate observations.")
if integrated.gm_well_id.nunique() != observations.gm_well_id.nunique():
    raise AssertionError("The integration changed the set of well IDs.")
features = [col for col in integrated.columns if col not in observations.columns]
missingness = integrated[features].isna().mean().sort_values(ascending=False)
print("Rows:", len(integrated), "| wells:", integrated.gm_well_id.nunique())
display(missingness.to_frame("missing_fraction"))

## 7. Create a new validation file

This cell refuses to overwrite an existing file. The preceding steps only build tables in memory; run this cell when ready to save the new validation output.

In [ ]:
OUTPUT_PATH = PROJECT_ROOT / "notebooks" / "merge" / "groundwater_ml_integration_validation.csv"
if OUTPUT_PATH.exists():
    raise FileExistsError(f"Will not overwrite {OUTPUT_PATH}")
integrated.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(integrated):,} rows to {OUTPUT_PATH}")

## Separate data and optional future methods

Karuna's geographically restricted A–R estimates are not extrapolated across the watershed. Surface-water nitrate, chemistry, and Thiessen well-depth estimates remain excluded.

The next cell contains two fully commented examples only. The nearest-reference result would be an estimate from a nearby reference well, not the target well's actual depth. Historical crop change is a future template for dated annual maps; it does not download or process data.

In [ ]:
# OPTIONAL EXAMPLE ONLY: do not run or add this estimated value to the ML table.
# Reference-well depth is estimated from a nearby reference well, not measured at the target well.
# reference_points = gpd.read_file(REFERENCE_WELLS_PATH).to_crs("EPSG:3310")
# target_points = well_points.to_crs("EPSG:3310")
# nearby_depth = gpd.sjoin_nearest(
#     target_points,
#     reference_points[["reference_well_id", "depth_m", "geometry"]],
#     how="left",
#     distance_col="reference_well_distance_m",
# )
# nearby_depth = nearby_depth.rename(
#     columns={"depth_m": "estimated_depth_from_nearest_reference_well_m"}
# )
#
# FUTURE TEMPLATE ONLY: compare dated annual crop maps if those data are approved.
# crops_2022 = pd.read_csv("crop_percentages_2022.csv")
# crops_2023 = pd.read_csv("crop_percentages_2023.csv")
# crop_change = crops_2023.merge(
#     crops_2022, on="well_id", suffixes=("_2023", "_2022"), validate="one_to_one"
# )
# crop_change["cropland_change_pct_points"] = (
#     crop_change["pct_cropland_2023"] - crop_change["pct_cropland_2022"]
# )

In [ ]:
import matplotlib.pyplot as plt

missing_pct = (
    integrated.isna().mean()
    .mul(100)
    .sort_values(ascending=True)
)

# Show only variables that actually have missing values
missing_pct = missing_pct[missing_pct > 0]

plt.figure(figsize=(10, 8))
missing_pct.plot(kind="barh")

plt.xlabel("Missing values (%)")
plt.ylabel("Variable")
plt.title("Missingness in final ML dataset")
plt.xlim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
missing_pct = missing_pct[missing_pct > 0]